In [1]:
import pandas as pd
import numpy as np

from sentence_transformers import SentenceTransformer
import hdbscan
import umap

from sklearn.feature_extraction.text import TfidfVectorizer, ENGLISH_STOP_WORDS

In [2]:
df = pd.read_csv("headlines_standardized.csv")

required_cols = ["headline", "headline_clean", "country", "label"]
assert all(col in df.columns for col in required_cols)

df = df.dropna(subset=["headline_clean"]).reset_index(drop=True)

print("Total headlines:", len(df))

Total headlines: 2969


In [4]:
model = SentenceTransformer("all-MiniLM-L6-v2")

texts = df["headline_clean"].astype(str).tolist()

embeddings = model.encode(
    texts,
    batch_size=32,
    show_progress_bar=True
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/93 [00:00<?, ?it/s]

In [5]:
from sklearn.cluster import KMeans

NUM_TOPICS = 15

kmeans = KMeans(
    n_clusters=NUM_TOPICS,
    random_state=42,
    n_init=20
)

df["topic_id"] = kmeans.fit_predict(embeddings)

print(df["topic_id"].value_counts().sort_index())

topic_id
0     177
1     188
2     175
3     128
4     245
5     308
6     196
7     107
8     167
9     190
10    211
11    193
12    193
13    173
14    318
Name: count, dtype: int64


In [6]:
from sklearn.feature_extraction.text import TfidfVectorizer, ENGLISH_STOP_WORDS
import numpy as np

custom_stopwords = list(
    ENGLISH_STOP_WORDS.union({
        "vs", "live", "update", "watch", "today",
        "breaking", "says", "said", "news", "report"
    })
)

def extract_keywords(texts, top_n=5):
    vectorizer = TfidfVectorizer(
        stop_words=custom_stopwords,
        ngram_range=(1, 2),
        max_features=5000
    )
    X = vectorizer.fit_transform(texts)
    scores = np.asarray(X.mean(axis=0)).ravel()
    top_idx = scores.argsort()[-top_n:][::-1]
    return [vectorizer.get_feature_names_out()[i] for i in top_idx]


In [7]:
topic_summary = []

for tid in range(NUM_TOPICS):
    texts = df[df.topic_id == tid]["headline_clean"]
    keywords = extract_keywords(texts, top_n=5)

    topic_summary.append({
        "topic_id": tid,
        "kw1": keywords[0],
        "kw2": keywords[1],
        "kw3": keywords[2],
        "kw4": keywords[3],
        "kw5": keywords[4],
        "num_headlines": len(texts)
    })

topic_df = pd.DataFrame(topic_summary).sort_values(
    "num_headlines", ascending=False
)

print(topic_df)


    topic_id         kw1           kw2        kw3              kw4       kw5  \
14        14        game         coach        nfl           injury  steelers   
5          5       india     bengaluru        bjp           mumbai       new   
4          4         ufc          2026     season              new     world   
10        10         man        police      court         arrested   charged   
6          6        2026            rs      price              new    demand   
11        11       court         rules        say       commission       ban   
12        12         day           new      quote        quote day      word   
9          9       death        killed       dies              man     crash   
1          1          wa           ice    missing              new   verizon   
0          0    transfer      football     portal  transfer portal     state   
2          2  australian     australia       open  australian open       day   
13        13       trump        trumps  

In [8]:
topic_name_map = {}

for _, row in topic_df.iterrows():
    topic_name_map[row["topic_id"]] = (
        f"{row['kw1']} / {row['kw2']} / {row['kw3']}"
    )

df["topic_name"] = df["topic_id"].map(topic_name_map)

In [9]:
df[df.topic_id == 3][["headline", "country", "label"]].head()

,headline,country,label
74,Winter rains trigger country's first parametri...,India,NaN
101,"Coldest Republic Day in 5 years, rain likely t...",India,NaN
147,Nearly 3.8 billion people may face extreme hea...,India,NaN
175,Decline of Himalayan rivers could trigger nucl...,India,NaN
264,Silver and Gold Sizzle on geopolitical heat,India,NaN


In [10]:
topics = {
    tid: group[["headline", "country", "label"]]
    for tid, group in df.groupby("topic_id")
}

In [11]:
topics[7]

,headline,country,label
56,Zilla Parishad polls: Congress candidate in La...,India,NaN
152,EU countries give final approval to Russian ga...,India,NaN
159,"Republican senator criticises Vance, Navarro &...",India,NaN
425,Iran's supreme leader admits 'thousands' kille...,India,NaN
427,US-Led Board Of Peace Announcement: Israel cri...,India,NaN
...,...,...,...
2866,Maduro's allies now ruling Venezuela reject US...,Australia,NaN
2868,Satellite images show damage from US strikes a...,Australia,NaN
2889,Man charged over ISIS-inspired New Year's Eve ...,Australia,NaN
2907,Trump says US will 'come to their rescue' if I...,Australia,NaN


In [12]:
df.to_csv("headlines_with_15_topics.csv", index=False)
topic_df.to_csv("topic_summary_15_topics.csv", index=False)

In [13]:
df[df["topic_id"] == 7]

,id,headline,country,label,headline_clean,topic_id,topic_name
56,57,Zilla Parishad polls: Congress candidate in La...,India,NaN,zilla parishad polls congress candidate in lat...,7,trump / venezuela / iran
152,153,EU countries give final approval to Russian ga...,India,NaN,eu countries give final approval to russian ga...,7,trump / venezuela / iran
159,160,"Republican senator criticises Vance, Navarro &...",India,NaN,republican senator criticises vance navarro tr...,7,trump / venezuela / iran
425,426,Iran's supreme leader admits 'thousands' kille...,India,NaN,irans supreme leader admits thousands killed i...,7,trump / venezuela / iran
427,428,US-Led Board Of Peace Announcement: Israel cri...,India,NaN,usled board of peace announcement israel criti...,7,trump / venezuela / iran
...,...,...,...,...,...,...,...
2866,2867,Maduro's allies now ruling Venezuela reject US...,Australia,NaN,maduros allies now ruling venezuela reject us ...,7,trump / venezuela / iran
2868,2869,Satellite images show damage from US strikes a...,Australia,NaN,satellite images show damage from us strikes a...,7,trump / venezuela / iran
2889,2890,Man charged over ISIS-inspired New Year's Eve ...,Australia,NaN,man charged over isisinspired new years eve at...,7,trump / venezuela / iran
2907,2908,Trump says US will 'come to their rescue' if I...,Australia,NaN,trump says us will come to their rescue if ira...,7,trump / venezuela / iran


In [ ]:
from transformers import pipeline

classifier = pipeline(
    "zero-shot-classification",
    model="facebook/bart-large-mnli"
)

config.json: 0.00B [00:00, ?B/s]

c:\Users\91638\AppData\Local\Programs\Python\Python310\lib\site-packages\huggingface_hub\file_download.py:130: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\91638\.cache\huggingface\hub\models--facebook--bart-large-mnli. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [21]:
classifier = pipeline(
    "zero-shot-classification",
    model="typeform/distilbert-base-uncased-mnli"
)

config.json:   0%|          | 0.00/776 [00:00<?, ?B/s]

c:\Users\91638\AppData\Local\Programs\Python\Python310\lib\site-packages\huggingface_hub\file_download.py:130: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\91638\.cache\huggingface\hub\models--typeform--distilbert-base-uncased-mnli. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/258 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

In [22]:
from tqdm import tqdm

BIAS_CLASSES = [
    "Neutral/Factual",
    "Sensational/Clickbait",
    "Human-Interest",
    "Promotional"
]

def batch_predict_bias(texts, batch_size=16):
    labels = []
    for i in tqdm(range(0, len(texts), batch_size)):
        batch = texts[i:i + batch_size]
        results = classifier(
            batch,
            candidate_labels=BIAS_CLASSES
        )
        labels.extend([r["labels"][0] for r in results])
    return labels

df["label"] = batch_predict_bias(df["headline"].tolist(), batch_size=16)

100%|██████████| 186/186 [07:32<00:00,  2.43s/it]


In [24]:
df.tail()

,id,headline,country,label,headline_clean,topic_id,topic_name
2964,2965,More than 120 business leaders call for Bondi ...,Australia,Human-Interest,more than 120 business leaders call for bondi ...,11,court / rules / say
2965,2966,More than 100 business leaders call for Bondi ...,Australia,Human-Interest,more than 100 business leaders call for bondi ...,11,court / rules / say
2966,2967,WA police searching for snorkeller missing off...,Australia,Human-Interest,wa police searching for snorkeller missing off...,1,wa / ice / missing
2967,2968,"Man, 34, charged with 66 offences after police...",Australia,Human-Interest,man 34 charged with 66 offences after police h...,10,man / police / court
2968,2969,Breaking: Several killed in New Year's Eve fir...,Australia,Human-Interest,breaking several killed in new years eve fire ...,9,death / killed / dies


In [25]:
df.to_csv("LLM_Labeled_Dataset.csv")
print("SAVED")

SAVED
